In [2]:
from pathlib import Path
from metasmith.python_api import Agent, Source, SshSource, Std, DataInstanceLibrary
from metasmith.python_api import DataTypeLibrary, Endpoint, WorkflowTask

dtypes, containers, transforms = Std()

smith = Agent(
    home = SshSource("fir", Path("/scratch/phyberos/metasmith")).AsSource(),
    setup_commands=[
        "module load StdEnv/2023",
        "module load apptainer/1.3.5",
    ]
)
# smith.Deploy()

In [ ]:
# https://training.nextflow.io/2.1.3/side_quests/workflows_of_workflows/#13-make-the-workflow-composable
# use this for array jobs

[]

In [4]:
lib = DataTypeLibrary()
lib.types = {"a": Endpoint(dtypes["short_reads"].properties|{"test"})}
lib.Save(Path("./cache/dlib_test"))

In [5]:
inputs = DataInstanceLibrary("./cache/new_sra_accessions.xgdb")
inputs.AddTypeLibrary("acc", lib)

DataTypeLibrary(schema='0.7.0', ontology=DataTypeOntology(name='EDAM', version='1.25', doi='https://doi.org/10.1093/bioinformatics/btt113', strict=False), types={'a': <{_:[test],data:Short sequence,format:Sequence file}:fb73gZZk>})

In [6]:
DataTypeLibrary.Load("./cache/dlib_test")["a"]

<{_:[test],data:Short sequence,format:Sequence file}:fb73gZZk>

In [7]:
# inputs = DataInstanceLibrary.Load("./cache/flye_inputs.xgdb")
inputs = DataInstanceLibrary.Load("./cache/long_reads.xgdb")
# inputs = DataInstanceLibrary.Load("./cache/asm.xgdb")
for p, n, e in inputs.Iterate():
    print(n, p, e, e.parents)

std::long_reads data.fastq <{data:Long sequence,format:Sequence file}:U6UcLaaP> set()


In [8]:
for p, n, t in containers.Iterate():
    with open(containers.location/p) as f:
        print(n)

std::oci_image_bakta
std::busco_annotation_script
std::busco_visualize_script
std::cazy_annotation_script
std::cazy_visualize_script
std::oci_image_diamond
std::oci_image_fasterq_dump
std::oci_image_fastqc
std::oci_image_filtlong
std::oci_image_flye
std::helpers_script
std::oci_image_kofamscan
std::oci_image_longqc
std::oci_image_megahit
std::oci_image_miniasm
std::oci_image_minimap2
std::oci_image_pilon
std::oci_image_prodigal
std::qc_table_script
std::oci_image_samtools
std::oci_image_script_runner
std::oci_image_trimmomatic


In [9]:
dtypes.types

{'short_reads': <{data:Short sequence,format:Sequence file}:M37WupEI>,
 'long_reads': <{data:Long sequence,format:Sequence file}:U6UcLaaP>,
 'reads': <{format:Sequence file}:wiUlXYOZ>,
 'self_mappings': <{format:Self-to-self mappings with minimap2}:ikG9D3QT>,
 'miniasm_estimate': <{format:Target bases estimated by miniasm}:FtqgfwUd>,
 'read_stats': <{data:Read statistics,format:Directory}:3BRou2gN>,
 'short_reads_accession': <{data:Accession number associated with short reads,format:Plaintext file}:4pOv1Zyo>,
 'long_reads_accession': <{data:Accession number associated with long reads,format:Plaintext file}:4HMb09gR>,
 'long_reads_filtered': <{data:Long reads filtered by filtlong,format:Filtered sequence file}:5wVVsQaA>,
 'short_reads_trimmed': <{data:Short reads trimmed by Trimmomatic,format:Trimmed sequence file}:0FOfqIHa>,
 'long_reads_assembly': <{data:Sequence assembly,format:Directory,from:Long reads}:5LXvuMiT>,
 'short_reads_assembly': <{data:Sequence assembly,format:Directory,fr

In [10]:
with open("./cache/slurm_account_fir") as f:
    SLURM_ACCOUNT = f.read()

In [11]:
task = smith.GenerateWorkflow(
    # seed=1,
    # max_refine=1024,
    # max_iter=1024,
    given      = [containers, inputs],
    transforms = [transforms],
    targets    = [dtypes["long_reads_assembly"]],
    # targets    = [dtypes["coding_sequences"]],
    # targets    = [dtypes["functional_annotations"]]
    # targets    = [dtypes["functional_annotations"].WithLineage([dtypes["long_reads"]])],
    # targets    = [dtypes["functional_annotations"].WithLineage([dtypes["hybrid_assembly"]])],
    # targets    = [dtypes["kofamscan_annotations"].WithLineage([dtypes["long_reads"]])],
    config = dict(
        nextflow = dict(
            preset="slurm",
            slurm_account=SLURM_ACCOUNT,
            cpus=8,
        )
    ),
)
for step in task.plan.steps:
    print(step.transform.name)
# task.RenderDAG("./cache/dag")

self_mappings
miniasm
filtlong_improved
flye


In [ ]:
WorkflowTask.Merge()

In [8]:
smith.StageWorkflow(task, on_exist="clear")

2025-09-02_16-50-15  | connecting to deployed agent
2025-09-02_16-50-15  | starting ssh to [fir]
2025-09-02_16-50-17  | starting relay service
 | > 2025-09-02_16-50-17  | connecting to relay as [03nq5VJVARmZ]


E| > 2025-09-02_16-50-17 E| relay server already running in [relay/connections]


2025-09-02_16-50-18 W| task already staged at [/scratch/phyberos/metasmith/runs/BPBh79aE]
2025-09-02_16-50-18 W| clearing previously staged task
2025-09-02_16-50-18  | sending metadata for workflow [BPBh79aE]
2025-09-02_16-50-32  | staging
 | > including dev binds
 | > 2025-09-02_16-50-32  | api call to [stage_workflow] with [{'task_key': 'BPBh79aE'}]
 | > 2025-09-02_16-50-32  | staging workflow [BPBh79aE] with [2] data libs and [1] transform libs
 | > 2025-09-02_16-50-32  | ex| /scratch/phyberos/metasmith
 | > 2025-09-02_16-50-32  | work [/ws/runs/BPBh79aE]
 | > 2025-09-02_16-50-32  | data [/msm_home/data]
 | > 2025-09-02_16-50-32  | external work [/scratch/phyberos/metasmith/runs/BPBh79aE]
 | > 2025-09-02_16-50-32  | external data [/scratch/phyberos/metasmith/data]
 | > 2025-09-02_16-50-32  | moving remote data libraries to [/msm_home/data]
 | > 2025-09-02_16-50-32  | using nextflow preset [slurm]
 | > 2025-09-02_16-50-32  | setting nextflow param [cpus] from config
 | > 2025-09-02_1

In [9]:
smith.RunWorkflow(task)

2025-09-02_17-00-34  | connecting to deployed agent
2025-09-02_17-00-34  | starting ssh to [fir]


2025-09-02_17-00-36  | starting relay service
 | > 2025-09-02_17-00-36  | connecting to relay as [FmsRa86XvZrT]


E| > 2025-09-02_17-00-36 E| relay server already running in [relay/connections]


2025-09-02_17-00-37  | triggering execution of [BPBh79aE]
2025-09-02_17-00-37  | closing connection


In [12]:
smith.CheckWorkflow(task)

2025-09-02_17-09-50  | connecting to deployed agent
2025-09-02_17-09-50  | starting ssh to [fir]
2025-09-02_17-09-51  | starting relay service


E| > 2025-09-02_17-09-51 E| relay server already running in [relay/connections]


 | > 2025-09-02_17-09-51  | connecting to relay as [vVbtiUgFF9xg]
 | > including dev binds
 | > 2025-09-02_17-09-52  | api call to [check_workflow] with [{'key': 'BPBh79aE'}]
 | > 2025-09-02_17-09-52  | searching for logs
 | > 2025-09-02_17-09-52  | found [1] runs
 | > 2025-09-02_17-09-52  |     1: [logs.2025-09-02_17-00-37]
 | > 2025-09-02_17-09-52  | here is the main log of the latest run [logs.2025-09-02_17-00-37]
 | > 2025-09-02_17-09-52  | >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
 | > 2025-09-02_17-09-52  | 
 | > including dev binds
 | > 2025-09-02_17-00-37  | api call to [run_workflow] with [{'key': 'BPBh79aE', 'log_dir': '_metasmith/logs.2025-09-02_17-00-37'}]
 | > 2025-09-02_17-00-37  | start time [2025-09-02_17-00-37]
 | > 2025-09-02_17-00-37  | running workflow [BPBh79aE] with preset [slurm]
 | > 2025-09-02_17-00-37  | loading agent metadata
 | > 2025-09-02_17-00-37  | workspace [/msm_home/runs/BPBh79aE]
 | > 2025-09-02_17-00-37  | external workspace 

In [9]:
# asm = DataInstanceLibrary("./cache/asm.xgdb")
# asm.Add(
#     items = [
#         ("/home/tony/workspace/projects/Toluene_resistence_evolution/data/gene_centric/nucleotide/fosmids.spades_meta.gt29kb.fna", "scadc.fna", "std::assembly"),
#     ],
# )
# asm.PruneTypes()

In [10]:
# fin = DataInstanceLibrary("./cache/flye_inputs.xgdb")
# fin.Add(
#     items = [
#         (Path("./cache/flye_in/miniasm_estimate").absolute(), "miniasm_estimate", "std::miniasm_estimate"),
#         (Path("./cache/flye_in/lr_ss10.fastq").absolute(), "lr_ss10.fastq", "std::long_reads_filtered"),
#     ],
# )
# fin.AddParentsTo("lr_ss10.fastq", [fin.Get("miniasm_estimate")])
# fin.PruneTypes()

In [11]:
# import os
# r = Path("/home/tony/workspace/tools/Metasmith/src/metasmith/std/containers")
# for f in r.iterdir():
#     os.system(f"""\
#         cd {r}
#         git mv {f.name} {f.name}.oci.uri
#     """)